# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [1]:
!pip install beir
!pip install rank_bm25 sentence-transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 1.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 4.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 58.2 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0

In [2]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

'../data/beir_datasets/scifact'

In [4]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

In [5]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [6]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [7]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [8]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [9]:
# ==========================================
# PARTE 2: Retrieval inicial con BM25
# ==========================================
from rank_bm25 import BM25Okapi
from beir.retrieval.evaluation import EvaluateRetrieval

# 1. Tokenización básica: Dividimos el texto del corpus por espacios
# Para mejorar, se podrían quitar stopwords, pero esto cumple con el baseline.
print("Entrenando BM25...")
tokenized_corpus = [str(text).split(" ") for text in df_corpus['text'].tolist()]
bm25 = BM25Okapi(tokenized_corpus)

# 2. Generar el ranking inicial para cada query
results_bm25 = {}
top_k_bm25 = 100 # Extraemos los 100 mejores para re-rankearlos después

print("Calculando scores BM25 para las queries...")
for idx, row in df_queries.iterrows():
    qid = str(row['query_id'])
    query = str(row['query'])
    tokenized_query = query.split(" ")
    
    # Obtener la puntuación de BM25 para la query contra todo el corpus
    doc_scores = bm25.get_scores(tokenized_query)
    
    # Obtener los índices de los 'top_k_bm25' documentos con mayor score
    top_n_indices = doc_scores.argsort()[::-1][:top_k_bm25]
    
    # Guardar en el formato que pide BEIR: dict[query_id][doc_id] = score
    results_bm25[qid] = {str(df_corpus.iloc[i]['doc_id']): float(doc_scores[i]) for i in top_n_indices}

# 3. Evaluar el baseline
evaluator = EvaluateRetrieval()
# Calculamos las métricas en el Top 10
ndcg_bm25, map_bm25, recall_bm25, precision_bm25 = evaluator.evaluate(qrels, results_bm25, k_values=[10])

print(f"\\nMétricas BM25 (Baseline):")
print(f"nDCG@10: {ndcg_bm25['NDCG@10']:.4f}")
print(f"Recall@10: {recall_bm25['Recall@10']:.4f}")

Entrenando BM25...
Calculando scores BM25 para las queries...
\nMétricas BM25 (Baseline):
nDCG@10: 0.5056
Recall@10: 0.6247


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [10]:
# ==========================================
# PARTE 3: Re-ranking con Cross-Encoder
# ==========================================
from sentence_transformers import CrossEncoder

# Cargamos un modelo ligero y eficiente para re-ranking
print("Cargando modelo Cross-Encoder...")
model_ce = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)

results_ce = {}

print("Re-rankeando con Cross-Encoder...")
for qid, docs in results_bm25.items():
    # Obtener el texto de la query
    query_text = df_queries[df_queries['query_id'] == qid]['query'].values[0]
    
    # Preparar los pares (query, documento) para pasarlos al modelo
    pairs = []
    doc_ids = list(docs.keys())
    
    for did in doc_ids:
        doc_text = df_corpus[df_corpus['doc_id'] == did]['text'].values[0]
        pairs.append((query_text, doc_text))
    
    # El modelo predice un score de relevancia para cada par
    scores = model_ce.predict(pairs)
    
    # Actualizamos los resultados con los nuevos scores semánticos
    results_ce[qid] = {doc_ids[i]: float(scores[i]) for i in range(len(doc_ids))}

print("¡Re-ranking Cross-Encoder completado!")

# Para identificar cambios de posición en el Top 10 (Ejemplo con la query 133 del Parte 1)
qid_ejemplo = "133"
if qid_ejemplo in results_bm25 and qid_ejemplo in results_ce:
    top10_bm25 = sorted(results_bm25[qid_ejemplo].items(), key=lambda x: x[1], reverse=True)[:10]
    top10_ce = sorted(results_ce[qid_ejemplo].items(), key=lambda x: x[1], reverse=True)[:10]
    
    print(f"\\nComparación Top 10 para la Query {qid_ejemplo}:")
    print("Doc IDs en BM25:", [d[0] for d in top10_bm25])
    print("Doc IDs en Cross-Encoder:", [d[0] for d in top10_ce])

Cargando modelo Cross-Encoder...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Re-rankeando con Cross-Encoder...
¡Re-ranking Cross-Encoder completado!
\nComparación Top 10 para la Query 133:
Doc IDs en BM25: ['26688294', '37964706', '5270265', '12785130', '45764440', '9507605', '5821617', '23076291', '29073751', '4399311']
Doc IDs en Cross-Encoder: ['16280642', '12640810', '6969753', '21295300', '9507605', '17934082', '86694016', '19752008', '9063688', '1410197']


## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [11]:
# ==========================================
# PARTE 4: Re-ranking con LTR (Random Forest)
# ==========================================
from sklearn.ensemble import RandomForestRegressor
import numpy as np

print("Extrayendo características para LTR...")
X = [] # Matriz de características (Features)
y = [] # Vector de etiquetas (Labels de relevancia)

# 1. Construir el dataset de entrenamiento extrayendo características de los resultados de BM25
for qid, docs in results_bm25.items():
    query_text = str(df_queries[df_queries['query_id'] == qid]['query'].values[0])
    q_len = len(query_text.split())
    
    for did, bm25_score in docs.items():
        doc_text = str(df_corpus[df_corpus['doc_id'] == did]['text'].values[0])
        d_len = len(doc_text.split())
        
        # Etiqueta: 1 si el documento es relevante según los 'qrels' originales, 0 si no lo es
        es_relevante = 1 if (qid in qrels and did in qrels[qid] and qrels[qid][did] > 0) else 0
        
        # Nuestras características: [Score BM25, Longitud del Doc, Longitud de Query]
        X.append([bm25_score, d_len, q_len])
        y.append(es_relevante)

# 2. Entrenar el modelo de Regresión
print("Entrenando modelo LTR (Random Forest)...")
rf_model = RandomForestRegressor(n_estimators=50, random_state=42)
rf_model.fit(X, y)

# 3. Generar el nuevo ranking con el modelo entrenado
results_ltr = {}
print("Re-rankeando con LTR...")
for qid, docs in results_bm25.items():
    query_text = str(df_queries[df_queries['query_id'] == qid]['query'].values[0])
    q_len = len(query_text.split())
    
    features = []
    doc_ids = list(docs.keys())
    
    for did in doc_ids:
        doc_text = str(df_corpus[df_corpus['doc_id'] == did]['text'].values[0])
        d_len = len(doc_text.split())
        features.append([docs[did], d_len, q_len])
        
    # Predecimos la probabilidad de relevancia usando el RF
    preds = rf_model.predict(features)
    
    # Guardamos los nuevos scores
    results_ltr[qid] = {doc_ids[i]: float(preds[i]) for i in range(len(doc_ids))}

print("¡Re-ranking LTR completado!")

Extrayendo características para LTR...
Entrenando modelo LTR (Random Forest)...
Re-rankeando con LTR...
¡Re-ranking LTR completado!


## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [12]:
# ==========================================
# PARTE 5: Evaluación Post Re-ranking
# ==========================================

# Ya teníamos el evaluador importado de BEIR
evaluator = EvaluateRetrieval()

# Evaluamos Cross-Encoder
ndcg_ce, map_ce, recall_ce, precision_ce = evaluator.evaluate(qrels, results_ce, k_values=[10])

# Evaluamos LTR
ndcg_ltr, map_ltr, recall_ltr, precision_ltr = evaluator.evaluate(qrels, results_ltr, k_values=[10])

# Resumen de resultados
import pandas as pd

resultados = {
    "Modelo": ["BM25 (Baseline)", "Cross-Encoder", "LTR (Random Forest)"],
    "nDCG@10": [ndcg_bm25['NDCG@10'], ndcg_ce['NDCG@10'], ndcg_ltr['NDCG@10']],
    "MAP@10": [map_bm25['MAP@10'], map_ce['MAP@10'], map_ltr['MAP@10']],
    "Recall@10": [recall_bm25['Recall@10'], recall_ce['Recall@10'], recall_ltr['Recall@10']]
}

df_resultados = pd.DataFrame(resultados)
print("\\n--- TABLA DE COMPARACIÓN FINAL ---")
display(df_resultados)

\n--- TABLA DE COMPARACIÓN FINAL ---


,Modelo,nDCG@10,MAP@10,Recall@10
0,BM25 (Baseline),0.50562,0.46273,0.62472
1,Cross-Encoder,0.62616,0.58890,0.72350
2,LTR (Random Forest),0.76463,0.76183,0.76183
